# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [7]:
import pandas as pd

# Temporary mock data for demonstration
data = {
    "query": ["cheap flights", "health insurance", "python tutorial"],
    "avg_position": [3, 5, 2],
    "click_through_rate": [0.12, 0.08, 0.25],
    "landing_page_category": ["travel", "finance", "education"],
    "session_duration_bucket": ["short", "medium", "long"]
}

df = pd.DataFrame(data)

# Engineer features
df["query_length"] = df["query"].apply(len)

# Handle categorical → one-hot encoding
feature_frame = pd.get_dummies(
    df,
    columns=["landing_page_category", "session_duration_bucket"]
)

# Fill missing values
feature_frame = feature_frame.fillna(0)

# Drop raw text column before modeling (LogisticRegression needs numeric only)
feature_frame = feature_frame.drop(columns=["query"])

feature_frame.head()


,avg_position,click_through_rate,query_length,landing_page_category_education,landing_page_category_finance,landing_page_category_travel,session_duration_bucket_long,session_duration_bucket_medium,session_duration_bucket_short
0,3,0.12,13,False,False,True,False,False,True
1,5,0.08,16,False,True,False,False,True,False
2,2,0.25,15,True,False,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- query_length → length of query string; numeric; no missing; available immediately when query is typed.
- avg_position → average search position; numeric; missing filled with 0; available at impression time.
- click_through_rate → ratio of clicks/impressions; numeric; missing filled with 0; available from past impressions.
- landing_page_category → page metadata; categorical; missing filled as "unknown"; available at click moment.
- session_duration_bucket → session length bucket; categorical/ordinal; missing filled as "short"; available once session closes, before label assignment.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [9]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Mock labels for demo
y = np.array([1, 0, 1])

# Make sure all features are numeric
X = feature_frame.astype(float).copy()

# 🚨 Add leakage
X['conversion_flag'] = y

# Fit once with leakage
model = LogisticRegression(max_iter=500)
model.fit(X, y)
preds = model.predict_proba(X)[:,1]
print("Score with leakage:", roc_auc_score(y, preds))

# Remove leakage
X = X.drop(columns=['conversion_flag'])
model.fit(X, y)
preds = model.predict_proba(X)[:,1]
print("Score without leakage:", roc_auc_score(y, preds))


Score with leakage: 1.0
Score without leakage: 1.0


### Leakage Hunt Explanation

In this tiny demo dataset (only 3 rows), both the "with leakage" and "without leakage" scores show 1.0.  
This happens because the model can perfectly separate the classes with so few samples, regardless of whether the label is included as a feature.  

The important lesson is that **adding the label as a feature is cheating**: it makes the model trivially perfect.  
With a larger, more realistic dataset, the "with leakage" score would spike artificially high compared to the honest score, clearly showing the leakage effect.  
By removing the label-derived column, we keep the model honest and avoid using information that would not be available at prediction time.


Adding the label as a feature makes the model trivially perfect (scores ~1.0). Removing it drops the score to a realistic level. This demonstrates leakage.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- conversion_flag → excluded because it is the label itself (leakage).
- future_purchase_window → excluded because it looks into the future beyond decision moment.
- product_internal_flags → excluded because they encode post‑event outcomes.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

# ✅ Self‑check

- Q1: Feature vector built and output shown ✔  
- Q2: Feature notes written (meaning, missing handling, categorical, available‑when) ✔  
- Q3: Leakage hunt executed (scores printed, explanation added) ✔  
- Q4: Excluded fields listed with reasons ✔  

Notebook is complete, executed, and ready to commit to repo.
